# InstaNexus notebook with figures

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from Bio import SeqIO
import json
import os
import logging
import numpy as np
from scipy.stats import mannwhitneyu

# Import all our custom pipeline modules
from instanexus import preprocessing
from instanexus import assembly
from instanexus import clustering
from instanexus import alignment
from instanexus import consensus
from instanexus import visualization
from instanexus import helpers

# Set up logging to see the pipeline's progress
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

In [ ]:
pd.options.display.max_colwidth = None

In [ ]:
os.chdir('../../../../')

print(f"Current working directory: {os.getcwd()}")

In [ ]:
FIGURES_DIR = Path("figures")
print(FIGURES_DIR)

In [ ]:
# Path to the new raw data you want to test
INPUT_CSV = "inputs/bsa.csv"

# Base folder for all results
BASE_OUTPUT_FOLDER = "outputs_notebook"

# Paths to your static database files$
METADATA_PATH = "json/sample_metadata.json"
CONTAMINANTS_PATH = "fasta/contaminants.fasta"

# --- 2. Define Pipeline Parameters ---
RUN_NAME = Path(INPUT_CSV).stem
REFERENCE_MODE = True
CHAIN = ""

# Filtering params
CONFIDENCE_THRESHOLD = 0.8
MASS_ERR_LIMIT = 20
MIN_LENGTH = 7
MAX_IRT_ERROR = 60
MIN_ENTROPY = 1
PROSIT_FILTER = True
FDR_THRESHOLD = 0.03
Z_SCORE_THRESHOLD = -0.5

# Assembly params
ASSEMBLY_MODE = "greedy"
KMER_SIZE = 6
MIN_OVERLAP = 3
SIZE_THRESHOLD = 10

MIN_IDENTITY = 1
MAX_MISMATCHES = 0

# Clustering params
MIN_SEQ_ID = 0.85
COVERAGE = 0.8

In [ ]:
print(RUN_NAME)

In [ ]:
base_output_folder = Path(BASE_OUTPUT_FOLDER) / RUN_NAME

# Build the unique experiment folder name
folder_name_parts = [f"{ASSEMBLY_MODE}"]

if CONFIDENCE_THRESHOLD is not None:
    folder_name_parts.append(f"c{CONFIDENCE_THRESHOLD}")

if "dbg" in ASSEMBLY_MODE:
    folder_name_parts.append(f"ks{KMER_SIZE}")

folder_name_parts.append(f"mo{MIN_OVERLAP}")
folder_name_parts.append(f"ts{SIZE_THRESHOLD}")

if REFERENCE_MODE:
    folder_name_parts.extend([f"mi{MIN_IDENTITY}", f"mm{MAX_MISMATCHES}"])

run_folder_name = "_".join(folder_name_parts)
experiment_folder = base_output_folder / run_folder_name

# --- Define all intermediate paths ---
cleaned_csv_path = experiment_folder / "cleaned.csv"
scaffolds_folder = experiment_folder / "scaffolds"
scaffolds_fasta_path = scaffolds_folder / "scaffolds.fasta"
clustering_folder = scaffolds_folder / "clustering"
cluster_fasta_folder = clustering_folder / "cluster_fasta"
alignment_folder = scaffolds_folder / "alignment"
consensus_folder = scaffolds_folder / "consensus"

# --- Define a Run ID for logging ---
run_id_str = f"[{RUN_NAME} @ {run_folder_name}]"

logger.info(f"Pipeline starting for run: {run_id_str}")
logger.info(f"All results will be saved to: {experiment_folder}")

In [ ]:
sample_metadata = preprocessing.get_sample_metadata(
    run=RUN_NAME, 
    chain=CHAIN, 
    json_path=METADATA_PATH
)

In [ ]:
proteases = sample_metadata["proteases"]
protein = sample_metadata["protein"]
protein_norm = preprocessing.normalize_sequence(protein)

In [ ]:
print(f"Sample uses proteases: {proteases}")

print(f"Protein sequence length: {len(protein)} amino acids")

print(f"Normalized protein sequence: {protein_norm}")

## 1. Esploratory data analysis

In [ ]:
original_data = pd.read_csv(INPUT_CSV)

In [ ]:
original_data.columns

### iRT EDA

In [ ]:
# show me the distribution of ion_match_intensity
sns.histplot(original_data['ion_match_intensity'], bins=50)
plt.xlabel("Ion Match Intensity")
plt.ylabel("Count")
plt.title("Distribution of Ion Match Intensity")
plt.show()  

In [ ]:
# show me the distribution of iRT
sns.histplot(original_data['iRT'], bins=50)
plt.xlabel("iRT")
plt.ylabel("Count")
plt.title("Distribution of iRT")
plt.show()  

In [ ]:
# show me the distribution of iRT
sns.histplot(original_data['is_missing_irt_error'], bins=50)
plt.xlabel("is_missing_irt_error")
plt.ylabel("Count")
plt.title("Distribution of is_missing_irt_error")
plt.show()  

In [ ]:
# show me the distribution of iRT
sns.histplot(original_data['iRT error'], bins=50)
plt.xlabel("iRT error")
plt.ylabel("Count")
plt.title("Distribution of iRT error")
plt.show()  

In [ ]:
# show me the distribution of iRT
sns.histplot(original_data['predicted iRT'], bins=50)
plt.xlabel("predicted iRT")
plt.ylabel("Count")
plt.title("Distribution of predicted iRT")
plt.show()


### Token EDA

beam 0 and instanovo log probabilities look the same

In [ ]:
original_data['instanovo_token_log_probabilities_beam_0'][:10]

In [ ]:
original_data['instanovo_token_log_probabilities'][:10]

In [ ]:
original_data['diffusion_token_log_probabilities'][:10]

In [ ]:
original_data['diffusion_predictions_beam_0'][:10]

In [ ]:
original_data['diffusion_token_log_probabilities'][:10]

In [ ]:
original_data['token_log_probs'][50:]

In [ ]:
original_data['instanovo_token_log_probabilities'][:10]

### FDR and PROSIT features

In [ ]:
original_data['is_missing_prosit_features'].value_counts()

## Instanovo + with rescoring

In [ ]:
instanovo_plus_data = pd.read_csv("_archive/csv/v3/bsa.csv")

In [ ]:
instanovo_plus_data.columns

In [ ]:
instanovo_plus_data['instanovo_token_log_probabilities'][:10]

In [ ]:
instanovo_plus_data['diffusion_predictions_beam_0'][50:]

In [ ]:
instanovo_plus_data['diffusion_log_probabilities_beam_0'][50:]

In [ ]:
instanovo_plus_data['token_log_probs'][:10]

In [ ]:
instanovo_plus_data['diffusion_token_log_probabilities'][:10]

## 2. Preprocessing

In [ ]:
data = original_data.copy()

In [ ]:
cols_to_keep = [
    'experiment_name',
    'prediction_untokenised',
    'instanovo_token_log_probabilities',
    'calibrated_confidence',    
    'psm_q_value',
    'delta_mass_ppm',
    'Mass Error',               
    'is_missing_prosit_features', 
    'ion_match_intensity',
    'ion_matches',
    'iRT',
    'iRT error',
    'is_missing_irt_error',
    'predicted iRT',
    'margin',
    'entropy',
    'z-score'
    ]

data = original_data[cols_to_keep].copy()

In [ ]:
data.rename(columns={'calibrated_confidence': 'conf'}, inplace=True)

In [ ]:
data.head(3)

In [ ]:
# drop columns spectrum_id and scan_number
# data.drop(columns=['spectrum_id', 'scan_number'], inplace=True)

In [ ]:
data["protease"] = data["experiment_name"].apply(
    lambda name: preprocessing.extract_protease(name, proteases)
)

protease_col = data.pop("protease")
data.insert(data.columns.get_loc("prediction_untokenised") + 1, "protease", protease_col)

data.head(3)

In [ ]:
# data = preprocessing.clean_dataframe(data)

data = data.dropna(subset=["prediction_untokenised"])

In [ ]:
data.head(3)

In [ ]:
data["cleaned_preds"] = data["prediction_untokenised"].apply(preprocessing.remove_modifications)

# move cleaned_preds next to prediction_untokenised
cleaned_preds_col = data.pop("cleaned_preds")
data.insert(data.columns.get_loc("prediction_untokenised") + 1, "cleaned_preds", cleaned_preds_col)

In [ ]:
data.head(3)

In [ ]:
cleaned_psms = data["cleaned_preds"].tolist()

In [ ]:
filtered_psms = preprocessing.filter_contaminants(
    cleaned_psms, RUN_NAME , CONTAMINANTS_PATH
)

In [ ]:
data = data[data["cleaned_preds"].isin(filtered_psms)]

In [ ]:
# drop column prediction_untokenised
data.drop(columns=['prediction_untokenised'], inplace=True)

In [ ]:
data.head(3)

In [ ]:
data.shape

In [ ]:
data["mapped"] = data["cleaned_preds"].apply(
    lambda x: "True" if x in protein_norm else "False"
)

In [ ]:
data.head()

In [ ]:
# filter all the rows that have cleaned_preds lenght less than 7 amino acids
data = data[data['cleaned_preds'].str.len() >= 7]

In [ ]:
data.shape

In [ ]:
def add_quantification_data(df_main, run_name, fdr_threshold, inputs_folder="inputs"):
    """
    Filters df_main by FDR, then looks for a quantification file ({run_name}_quant_scores.csv).
    Merges the abundance data into the filtered dataframe.
    """
    # 1. FILTRO FDR (Lo facciamo subito, qui dentro)
    if fdr_threshold is not None:
        if "psm_q_value" in df_main.columns:
            initial_len = len(df_main)
            df_main = df_main[df_main['psm_q_value'] <= fdr_threshold].copy()
            logger.info(f"FDR Filter applied inside merge function: {initial_len} -> {len(df_main)} rows (<= {fdr_threshold})")
        else:
            logger.warning("FDR threshold provided but 'psm_q_value' column missing. Skipping filter.")

    quant_file_name = f"{run_name}_quant_scores.csv"
    quant_file_path = Path(inputs_folder) / quant_file_name
    
    if not quant_file_path.exists():
        logger.warning(f"Quantification file NOT FOUND: {quant_file_path}")
        logger.warning("Skipping abundance merging. 'peptide_abundance' will be missing.")
        return df_main

    logger.info(f"Found quantification file: {quant_file_path}")
    
    try:
        df_quant = pd.read_csv(quant_file_path)
        
        if "cleaned_preds" not in df_quant.columns or "total_abundance_norm" not in df_quant.columns:
            logger.warning(f"Quantification file format error. Missing columns in {quant_file_path}")
            return df_main

        # 2. Raggruppa e somma le abbondanze dal file quant
        df_quant_summed = df_quant.groupby('cleaned_preds', as_index=False)['total_abundance_norm'].sum()
        
        # 3. Rinomina
        df_quant_summed.rename(columns={'total_abundance_norm': 'peptide_abundance'}, inplace=True)
        
        # 4. Merge (Left Join su df_main che è ORA filtrato)
        df_merged = pd.merge(df_main, df_quant_summed, on='cleaned_preds', how='left')
        
        # 5. Riempi i buchi
        df_merged['peptide_abundance'] = df_merged['peptide_abundance'].fillna(0)
        
        logger.info(f"Quantification data merged successfully. Output rows: {len(df_merged)}")
        return df_merged

    except Exception as e:
        logger.error(f"Error merging quantification data: {e}")
        return df_main

In [ ]:
data_abundance = add_quantification_data(data, RUN_NAME, 0.1)

In [ ]:
data_abundance

In [ ]:
from instanexus.assembly import Assembler

In [ ]:
sequences = data_abundance['cleaned_preds'].tolist()

In [ ]:
assembler = Assembler(
    mode="multimodal_dbg",
    kmer_size=7,
    min_overlap=3,
    size_threshold=10,
    min_weight=2
)

In [ ]:
print(f"Starting Multimodal Assembly on {len(sequences)} peptides...")

In [ ]:
scaffolds = assembler.run(sequences=sequences, df_full=data_abundance)

In [ ]:
len(scaffolds)

In [ ]:
scaffolds

In [ ]:
# show me the distribution of the lenght of the list scaffolds
scaffold_lengths = [len(seq) for seq in scaffolds]
sns.histplot(scaffold_lengths, bins=30)
plt.xlabel("Scaffold Length")
plt.ylabel("Count")
plt.title("Distribution of Scaffold Lengths")
plt.show()  

In [ ]:
print(f"\nDone. Generated {len(scaffolds)} scaffolds.")

for i, seq in enumerate(scaffolds[:5]):
    print(f">scaffold_{i} (len {len(seq)})\n{seq}")

In [ ]:
def add_quantification_data(df_main, run_name, inputs_folder="inputs"):
    """
    Looks for a quantification file ({run_name}_quant_scores.csv) in inputs_folder.
    If found, merges the abundance data into the main dataframe.
    """
    quant_file_name = f"{run_name}_quant_scores.csv"
    quant_file_path = Path(inputs_folder) / quant_file_name
    
    if not quant_file_path.exists():
        logger.warning(f"Quantification file NOT FOUND: {quant_file_path}")
        logger.warning("Skipping abundance merging. 'peptide_abundance' will be missing.")
        return df_main

    logger.info(f"Found quantification file: {quant_file_path}")
    
    try:
        df_quant = pd.read_csv(quant_file_path)
        
        # Check required columns
        if "cleaned_preds" not in df_quant.columns or "total_abundance_norm" not in df_quant.columns:
            logger.warning(f"Quantification file format error. Missing columns in {quant_file_path}")
            return df_main

        # 1. Group by peptide (summing abundances if same peptide appears multiple times)
        # Nota: Non facciamo il log qui. Lasciamo il valore grezzo (sommato) 
        # perché l'assembler farà il log10 internamente.
        df_quant_summed = df_quant.groupby('cleaned_preds', as_index=False)['total_abundance_norm'].sum()
        
        # 2. Rename column for consistency with Assembler
        df_quant_summed.rename(columns={'total_abundance_norm': 'peptide_abundance'}, inplace=True)
        
        # 3. Merge with main dataframe
        # Left join: keep all PSMs, add abundance where available
        df_merged = pd.merge(df_main, df_quant_summed, on='cleaned_preds', how='left')
        
        # Fill NaNs with 0 for peptides that didn't have quantification
        df_merged['peptide_abundance'] = df_merged['peptide_abundance'].fillna(0)
        
        logger.info(f"Quantification data merged successfully. Rows with abundance: {len(df_quant_summed)}")
        return df_merged

    except Exception as e:
        logger.error(f"Error merging quantification data: {e}")
        return df_main

In [ ]:
data_with_protein_abundance = add_quantification_data(data, RUN_NAME)

In [ ]:
data_with_protein_abundance["peptide_abundance"].describe()

In [ ]:
def analyze_abundance_by_fdr(main_data, sample_name, fdr_threshold, inputs_folder="inputs"):
    """
    Analyzes peptide abundance by filtering for FDR, merging quantitative data,
    performing statistical analysis, and generating plots.

    Args:
        main_data (pd.DataFrame): The main dataframe containing psm_q_value and mapped columns.
        sample_name (str): The sample name (e.g., 'BSA') used to locate the CSV file.
        fdr_threshold (float): The FDR threshold (e.g., 0.05).
        inputs_folder (str): Folder where the *_quant_scores.csv files are located.

    Returns:
        pd.DataFrame: The final merged and processed dataframe.
    """
    
    print(f"\n{'='*60}")
    print(f"SAMPLE ANALYSIS: {sample_name} | FDR < {fdr_threshold}")
    print(f"{'='*60}")

    data_fdr = main_data[main_data['psm_q_value'] < fdr_threshold].copy()
    
    if data_fdr.empty:
        print(" No data found with this FDR threshold.")
        return None

    print(f"Rows in filtered dataset (FDR < {fdr_threshold}): {len(data_fdr)}")

    quant_file_path = os.path.join(inputs_folder, f"{sample_name}_quant_scores.csv")
    
    if not os.path.exists(quant_file_path):
        print(f" Error: File not found: {quant_file_path}")
        return None
        
    df_quant = pd.read_csv(quant_file_path)

    list_cleaned_psms = data_fdr['cleaned_preds'].tolist()
    df_quant_filtered = df_quant[df_quant['cleaned_preds'].isin(list_cleaned_psms)].copy()

    df_quant_summed = df_quant_filtered.groupby('cleaned_preds', as_index=False)['total_abundance_norm'].sum()
    
    df_quant_summed['log_total_norm'] = np.log10(df_quant_summed['total_abundance_norm'].replace(0, np.nan))

    data_final = pd.merge(
        data_fdr, 
        df_quant_summed[['cleaned_preds', 'log_total_norm']], 
        on='cleaned_preds', 
        how='left'
    )

    data_final['mapped'] = data_final['mapped'].astype(str).str.lower() == 'true'

    df_analysis = data_final.dropna(subset=['log_total_norm']).copy()
    
    mapped_counts = df_analysis['mapped'].value_counts()
    print(f"Samples with valid abundance: {len(df_analysis)}")
    print(f" - Mapped TRUE:  {mapped_counts.get(True, 0)}")
    print(f" - Mapped FALSE: {mapped_counts.get(False, 0)}")

    if len(df_analysis) < 5 or mapped_counts.get(True, 0) < 2 or mapped_counts.get(False, 0) < 2:
        print("Insufficient data for statistical analysis/plotting.")
        return data_final

    plt.figure(figsize=(10, 6))
    
    sns.boxplot(x='mapped', y='log_total_norm', data=df_analysis, showfliers=False, palette="Set2")
    
    if len(df_analysis) < 2000:
        sns.stripplot(x='mapped', y='log_total_norm', data=df_analysis, color=".25", alpha=0.5)

    plt.title(f'Abundance Distribution ({sample_name}, FDR < {fdr_threshold})')
    plt.ylabel('Log10 Total Abundance')
    plt.xlabel('Is Mapped?')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

    group_true = df_analysis[df_analysis['mapped'] == True]['log_total_norm']
    group_false = df_analysis[df_analysis['mapped'] == False]['log_total_norm']

    print("\n--- Descriptive Statistics ---")
    print(f"Median TRUE:  {group_true.median():.2f} (n={len(group_true)})")
    print(f"Median FALSE: {group_false.median():.2f} (n={len(group_false)})")

    stat, p_value = mannwhitneyu(group_true, group_false, alternative='two-sided')

    print("\n--- Mann-Whitney U Test ---")
    print(f"P-value: {p_value:.5e}")

    if p_value < 0.05:
        print("SIGNIFICANT RESULT")
        direction = "HIGHER" if group_true.median() > group_false.median() else "LOWER"
        print(f"   Mapped peptides have {direction} abundance.")
    else:
        print("NOT SIGNIFICANT")

    return data_final

In [ ]:
fdr_levels = [0.2, 0.1, 0.05, 0.01]

results = {}

for fdr in fdr_levels:
    df_result = analyze_abundance_by_fdr(data, RUN_NAME, fdr)
    results[fdr] = df_result

#### HCP contaminations

E.coli can still be present in the target protein we are analysing. 
E.coli is used to produce our samples proteins by growing 

In [ ]:
# show me only mapped true subset and then we can plot the confidence distributions

plt.figure(figsize=(10, 6))
plt.hist(unmapped_conf, bins=50, alpha=0.7, label='Incorrectly mapped predictions', color='red')
plt.hist(mapped_conf, bins=50, alpha=0.7, label='Correctly mapped predictions', color='blue')
plt.title('Confidence Score Distributions')
plt.xlabel('Confidence Score')
plt.ylabel('Density')
plt.legend()
plt.show()

In [ ]:
# show me only mapped true subset and then we can plot the confidence distributions

plt.figure(figsize=(10, 6))
plt.hist(unmapped_conf, bins=50, alpha=0.7, label='Incorrectly mapped predictions', color='red')
plt.hist(mapped_conf, bins=50, alpha=0.7, label='Correctly mapped predictions', color='blue')
plt.yscale('log')
plt.title(f'Confidence Score Distributions in sample {RUN_NAME}')
plt.xlabel('Confidence Score')
plt.ylabel('Density')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(mapped_conf, bins=50, alpha=0.7, label='Correctly mapped predictions', color='blue')
plt.title(f'Confidence score distributions in sample {RUN_NAME}')
plt.xlabel('Confidence Score')
plt.ylabel('Density')
plt.legend()
plt.show()

## Data cleaning with multiple features

In [ ]:
data.head()

In [ ]:
def optional_filtering(
        dataframe,
        conf_threshold: float,
        mass_error_limit: int,
        min_length: int,
        max_irt_error: int,
        min_entropy: int,
        remove_missing_prosit: bool,
        fdr_threshold: float,
        z_score_threshold: float,
        verbose: bool
):
    df_filtered = dataframe.copy()
    
    if conf_threshold is not None:
        df_filtered = df_filtered[df_filtered['conf'] >= conf_threshold]
        if verbose:
            print(f"Filtered by confidence >= {conf_threshold}: {df_filtered.shape[0]} rows remaining.")
    
    if mass_error_limit is not None:
        df_filtered = df_filtered[abs(df_filtered['Mass Error']) <= mass_error_limit]
        if verbose:
            print(f"Filtered by mass error <= {mass_error_limit} ppm: {df_filtered.shape[0]} rows remaining.")
    
    if min_length is not None:
        df_filtered = df_filtered[df_filtered['cleaned_preds'].str.len() >= min_length]
        if verbose:
            print(f"Filtered by minimum length >= {min_length}: {df_filtered.shape[0]} rows remaining.")
    
    if max_irt_error is not None:
        df_filtered = df_filtered[abs(df_filtered['iRT error']) <= max_irt_error]
        if verbose:
            print(f"Filtered by iRT error <= {max_irt_error}: {df_filtered.shape[0]} rows remaining.")
    
    if min_entropy is not None:
        df_filtered = df_filtered[df_filtered['entropy'] >= min_entropy]
        if verbose:
            print(f"Filtered by entropy >= {min_entropy}: {df_filtered.shape[0]} rows remaining.")
    
    if remove_missing_prosit:
        # if remove_missing_prosit is True, filter out rows where is_missing_prosit_features is True
        df_filtered = df_filtered[~df_filtered['is_missing_prosit_features']]
        if verbose:
            print(f"Removed missing Prosit features: {df_filtered.shape[0]} rows remaining.")

    if fdr_threshold is not None:
        df_filtered = df_filtered[df_filtered['psm_q_value'] <= fdr_threshold]
        if verbose:
            print(f"Filtered by FDR <= {fdr_threshold}: {df_filtered.shape[0]} rows remaining.")
    if z_score_threshold is not None:
        # keep only rows where absolute value of z-score is greater than or equal to threshold
        df_filtered = df_filtered[abs(df_filtered['z-score']) >= z_score_threshold]
        if verbose:
            print(f"Filtered by |z-score| >= {z_score_threshold}: {df_filtered.shape[0]} rows remaining.")

    return df_filtered

### filtering parameter explanations


MASS_ERR_LIMIT,

MAX_IRT_ERROR,
MIN_ENTROPY,
PROSIT_FILTER,
FDR_THRESHOLD,


**PROSIT_FILTER**
1. False (Good): Prosit successfully generated a theoretical spectrum and retention time for this peptide. The system was able to compare the de novo prediction against the Prosit prediction to calculate quality scores (like spectral angle, iRT error, etc.).
2. True (Bad): Prosit failed to generate predictions for this peptide sequence. Therefore, all columns dependent on Prosit (like prosit_intensity, iRT error, ion_matches) are likely empty, zero, or unreliable.

**MASS ERROR**

In [ ]:
data_filtered = optional_filtering(
    dataframe=data,
    conf_threshold=CONFIDENCE_THRESHOLD,
    mass_error_limit=MASS_ERR_LIMIT,
    min_length=MIN_LENGTH,
    max_irt_error=MAX_IRT_ERROR,
    min_entropy=MIN_ENTROPY,
    remove_missing_prosit=PROSIT_FILTER,
    fdr_threshold=FDR_THRESHOLD,
    z_score_threshold=Z_SCORE_THRESHOLD,
    verbose=True
)

In [ ]:
data_filtered.head(3)

In [ ]:
# show me the lowest value of "conf"
print(data_filtered['conf'].min())

In [ ]:
mapped_filtered = data_filtered[data_filtered['mapped'] == "True"]['conf']
unmapped_filtered = data_filtered[data_filtered['mapped'] == "False"]['conf']

In [ ]:
print(mapped_filtered.shape[0])
print(unmapped_filtered.shape[0])

In [ ]:
# calculate FDR considering (mapped_filtered.shape[0]) / (mapped_filtered.shape[0] + unmapped_filtered.shape[0])
fdr_calculated = unmapped_filtered.shape[0] / (mapped_filtered.shape[0] + unmapped_filtered.shape[0])
print(f"Calculated FDR after filtering: {fdr_calculated:.4f}")

In [ ]:
# show me only mapped true subset and then we can plot the confidence distributions

plt.figure(figsize=(10, 6))
plt.hist(unmapped_filtered, bins=50, alpha=0.7, label='Incorrectly mapped predictions', color='orange')
plt.hist(mapped_filtered, bins=50, alpha=0.7, label='Correctly mapped predictions', color='green')
plt.yscale('log')
plt.title(f'Confidence Score Distributions in sample {RUN_NAME}')
plt.xlabel('Confidence Score')
plt.ylabel('Density')
plt.legend()
plt.show()

## HCP filtering

In [ ]:
#reindex the dataframe
data_filtered = data_filtered.reset_index(drop=True)

In [ ]:
data_filtered.head(10)

In [ ]:
data_filtered.shape

In [ ]:
data_filtered.to_csv(f"outputs_notebook/{RUN_NAME}_accepted_hits.csv", index=False)

print(f"Accepted hits saved to: outputs_notebook/{RUN_NAME}_accepted_hits.csv")


### Check scaffolds at different FDR

In [ ]:
fdr_thresholds = [0.01, 0.05, 0.10, 0.20]

output_stats_folder = "outputs_notebook/fdr_stats"
os.makedirs(output_stats_folder, exist_ok=True)

In [ ]:
from instanexus.assembly import Assembler 

assembler = Assembler(
    mode=ASSEMBLY_MODE,
    kmer_size=KMER_SIZE,
    min_overlap=MIN_OVERLAP,
    size_threshold=SIZE_THRESHOLD,
    min_identity=MIN_IDENTITY,
    max_mismatches=MAX_MISMATCHES
)

In [ ]:
fdr_thresholds = [0.01, 0.05, 0.10, 0.20]
coverage_results = []
scaffold_counts = []

In [ ]:
output_stats_folder = "outputs_notebook/fdr_stats"
os.makedirs(output_stats_folder, exist_ok=True)

In [ ]:
data.head(10)

In [ ]:
for fdr in fdr_thresholds:
    
    print(f"\n>>> Processing FDR: {fdr*100}%")
    
    subset_df = optional_filtering(
        dataframe=data, 
        fdr_threshold=fdr,
        mass_error_limit=MASS_ERR_LIMIT,
        min_length= MIN_LENGTH,
        max_irt_error=MAX_IRT_ERROR, 
        min_entropy=MIN_ENTROPY,
        remove_missing_prosit=PROSIT_FILTER,
        conf_threshold=None, 
        z_score_threshold=Z_SCORE_THRESHOLD,
        verbose=False
    )

    input_sequences = subset_df['cleaned_preds'].dropna().tolist()
    
    if not input_sequences:
        print(f"  No peptides remaining.")
        coverage_results.append(0)
        scaffold_counts.append(0)
        continue

    # ASSEMBLY
    try:
        scaffolds_sequences = assembler.run(sequences=input_sequences)
    except Exception as e:
        print(f"  Assembly failed: {e}")
        coverage_results.append(0)
        scaffold_counts.append(0)
        continue
    
    if not scaffolds_sequences:
        print(f"  No scaffolds produced.")
        coverage_results.append(0)
        scaffold_counts.append(0)
        continue

    mapped_sequences = visualization.process_protein_contigs_scaffold(
        assembled_contigs=scaffolds_sequences,
        target_protein=protein_norm,
        max_mismatches=MAX_MISMATCHES,
        min_identity=MIN_IDENTITY
    )
    
    if not mapped_sequences:
        print(f"  {len(scaffolds_sequences)} scaffolds produced, but NONE mapped.")
        coverage_results.append(0)
        scaffold_counts.append(len(scaffolds_sequences))
        continue

    # STATISTICS
    df_mapped = visualization.create_dataframe_from_mapped_sequences(data=mapped_sequences)
    
    stats = helpers.compute_assembly_statistics(
        df=df_mapped,
        sequence_type=f"scaffolds_fdr_{int(fdr*100)}",
        output_folder=output_stats_folder,
        reference=protein_norm,
        fdr_threshold=fdr 
    )
    
    cov = stats["coverage"] * 100 
    coverage_results.append(cov)
    scaffold_counts.append(len(scaffolds_sequences))

    print(f"  Result: Input {len(input_sequences)} -> Scaffolds {len(scaffolds_sequences)} -> Coverage {cov:.2f}%")

plt.figure(figsize=(10, 6))
labels = [f"{int(x*100)}%" for x in fdr_thresholds]

plt.plot(labels, coverage_results, marker='o', linestyle='-', color='#2c7bb6', linewidth=2, markersize=10, label='Coverage')

for i, cov in enumerate(coverage_results):
    plt.annotate(f"{cov:.1f}%", (labels[i], cov), textcoords="offset points", xytext=(0,10), ha='center', fontweight='bold')
    plt.annotate(f"({scaffold_counts[i]} scaf)", (labels[i], cov), textcoords="offset points", xytext=(0,-25), ha='center', fontsize=9, color='gray')

plt.title(f'Scaffold Coverage vs. FDR Threshold', fontsize=14)
plt.xlabel('FDR Threshold', fontsize=12)
plt.ylabel('Scaffold Sequence Coverage (%)', fontsize=12)
plt.ylim(0, 110)
plt.grid(True, alpha=0.6)

plt.show()

## 3. Assembly